# MVP v0.3.2: Multi-Policy Positive and Negative Test

**Date:** 2026-03-12  
**Builds on:** MVP v0.2.5 (reward model fix) + v0.2.4.2 (target+expert diffuser)

## Goal

Evaluate the full OPE pipeline across **8 target policies** spanning 0–90% success rate.
This is the first multi-policy evaluation — testing whether OPE estimates correlate with
true policy value (Spearman rank correlation).

## Positive Test
OPE estimates should rank policies correctly: higher oracle SR → higher OPE estimate.

## Negative Test
A 0% SR policy should get OPE ≈ 0. A 90% SR policy should get OPE ≈ 0.9.
OPE should not collapse all policies to the same estimate.

## Pipeline (per target policy)

1. Collect 50 rollouts from target policy (or load if cached)
2. Load rollouts + 200 expert demos → chunk for diffuser training
3. Train chunk diffuser (50 epochs)
4. Train BC_Gaussian behavior policy
5. Build target policy scorer
6. Guided stitching (best config: full_0.2_r0.25)
7. Score synthetic trajectories → OPE estimate

Then: multi-policy metrics (Spearman ρ, regret@k, scatter plot).

In [1]:
import sys, os
import importlib
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm
from scipy import stats
import time

# Project root
PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

# SOPE imports
from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

# Robomimic imports
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

# Our imports
import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer
from latent_sope.robomimic_interface.collect import collect_rollouts

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.l

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda


## Configuration

In [2]:
# ── Obs keys (sorted, matching robomimic convention & LowDimConcatEncoder layout) ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

# Full dims
STATE_DIM = 19   # object(10) + eef_pos(3) + eef_quat(4) + gripper_qpos(2)
ACTION_DIM = 7   # full robomimic action space
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 26

# ── Chunk config ──
CHUNK_SIZE = 4
FRAME_STACK = 1
STRIDE = 2

# ── Diffusion config ──
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False

# ── Training config ──
TRAIN_EPOCHS = 25
BATCH_SIZE = 64
LR = 3e-4
GRAD_CLIP = 1.0

# ── OPE config ──
NUM_TARGET_ROLLOUTS = 20
NUM_SYNTHETIC_TRAJS = 20
T_GEN = 60
GAMMA = 1.0

# ── Reward ──
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84

# ── Guidance config (best from v0.2.4: full_0.2_r0.25) ──
K_GUIDE = 1
NORMALIZE_GRAD = True
USE_ADAPTIVE = False
CLAMP = False
L_INF = 1.0
BEST_ACTION_SCALE = 0.2
BEST_RATIO = 0.25

# ── BC_Gaussian config ──
BC_EPOCHS = 500
BC_BATCH = 256
BC_HIDDEN = 256

# ── Expert demos ──
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"

# ── Paths ──
CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"
ROLLOUT_BASE = PROJECT_ROOT / "rollouts"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-12"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 4 Target policies (spanning 0–90% SR, quick test) ──
POLICIES = {
    "50demos_epoch10":  CKPT_BASE / "lift_diffusion_50demos"  / "20260311134204",
    "200demos_epoch10": CKPT_BASE / "lift_diffusion_200demos" / "20260311141036",
    "10demos_epoch20":  CKPT_BASE / "lift_diffusion_10demos"  / "20260311115828",
    "200demos_epoch40": CKPT_BASE / "lift_diffusion_200demos" / "20260311141036",
}

# Extract epoch number from key
def get_epoch(key):
    return int(key.split("epoch")[1])

# Oracle results
ORACLE_JSON = RESULTS_DIR / "oracle_eval_all_checkpoints.json"

print(f"Evaluating {len(POLICIES)} target policies:")
for key in POLICIES:
    print(f"  {key}")

Evaluating 4 target policies:
  50demos_epoch10
  200demos_epoch10
  10demos_epoch20
  200demos_epoch40


## Step 0: Load Oracle Values for All Target Policies

In [3]:
with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)

oracle_values = {}
for key in POLICIES:
    od = oracle_data[key]
    oracle_values[key] = {
        "mean_return": od["mean_return"],
        "std_return": od["std_return"],
        "success_rate": od["success_rate"],
    }
    print(f"  {key:>20s}: V^pi={od['mean_return']:.4f}, SR={od['success_rate']*100:.1f}%")

       50demos_epoch10: V^pi=0.0000, SR=0.0%
      200demos_epoch10: V^pi=0.1800, SR=18.0%
       10demos_epoch20: V^pi=0.5200, SR=52.0%
      200demos_epoch40: V^pi=0.9000, SR=90.0%


## Step 1: Load Expert Demos (shared across all target policies)

In [4]:
expert_data = []
expert_states_list = []
expert_actions_list = []

with h5py.File(DEMO_HDF5, "r") as f:
    demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    for dk in tqdm(demo_keys, desc="Loading expert demos"):
        demo = f[f"data/{dk}"]
        obs_arrays = [demo["obs"][k][:].astype(np.float32) for k in OBS_KEYS]
        states = np.concatenate(obs_arrays, axis=-1)  # (T, 19)
        actions = demo["actions"][:].astype(np.float32)  # (T, 7)
        rewards = demo["rewards"][:].astype(np.float32)
        
        episode = {
            "states": states[:-1],
            "actions": actions[:-1],
            "rewards": rewards[:-1],
            "next-states": states[1:],
        }
        expert_data.append(episode)
        expert_states_list.append(states)
        expert_actions_list.append(actions)

print(f"Loaded {len(expert_data)} expert demos")

Loading expert demos:   0%|          | 0/200 [00:00<?, ?it/s]

Loading expert demos:  35%|███▌      | 70/200 [00:00<00:00, 694.15it/s]

Loading expert demos:  72%|███████▎  | 145/200 [00:00<00:00, 722.59it/s]

Loading expert demos: 100%|██████████| 200/200 [00:00<00:00, 729.30it/s]

Loaded 200 expert demos


## Helper Functions

BCGaussian, trajectory generation, and scoring — same as v0.2.5.

In [5]:
class BCGaussian(nn.Module):
    """Simple BC policy with Gaussian output: pi(a|s) = N(mu(s), sigma(s))."""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        h = self.net(state)
        mean = self.mean_head(h)
        log_std = self.log_std_head(h).clamp(-5, 2)
        return mean, log_std
    
    def log_prob(self, state, action):
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        return -0.5 * (((action - mean) / std) ** 2 + 2 * log_std + math.log(2 * math.pi)).sum(dim=-1)
    
    def grad_log_prob(self, state, action):
        with torch.no_grad():
            mean, log_std = self.forward(state)
            std = torch.exp(log_std)
            return -(action - mean) / (std ** 2)
    
    def grad_log_prob_chunk(self, states, actions):
        B, T, _ = states.shape
        s_flat = states.reshape(B * T, -1)
        a_flat = actions.reshape(B * T, -1)
        grad_flat = self.grad_log_prob(s_flat, a_flat)
        return grad_flat.reshape(B, T, -1)


def get_initial_states_from_data(data, num_samples, device):
    all_initial = np.array([ep["states"][0] for ep in data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def score_trajectories_gt(trajectories, cube_z_index, threshold):
    """Episode-level binary reward (v0.2.5 fix): 1.0 if cube lifted at any step."""
    B, T, D = trajectories.shape
    cube_z = trajectories[:, :, cube_z_index]
    successes = np.any(cube_z > threshold, axis=1)
    returns = successes.astype(np.float32)
    return returns, successes


def generate_trajectories_full_guidance(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim,
    chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    action_scale=0.0, ratio=0.5,
    k_guide=1, normalize_grad=True,
    use_adaptive=False, clamp=False, l_inf=1.0,
):
    guided = (target_scorer is not None and action_scale > 0)
    use_neg_grad = (behavior_scorer is not None and ratio > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    n_timesteps = diffusion_model.n_timesteps

    padded = torch.cat([
        initial_states,
        torch.zeros(batch_size, action_dim, device=device)
    ], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]

    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0
    n_iterations = 0

    while total_generated < t_gen:
        n_iterations += 1
        steps_remaining = t_gen - total_generated
        shape = (batch_size, chunk_size, transition_dim)

        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)

            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)

            if guided:
                model_mean = unnormalize_fn(model_mean)

                for _k in range(k_guide):
                    states_chunk = model_mean[:, :, :state_dim]
                    actions_chunk = model_mean[:, :, state_dim:]

                    target_grad = target_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)

                    if use_neg_grad:
                        behavior_grad = behavior_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)

                    if normalize_grad:
                        eps = 1e-6
                        target_norm = target_grad.norm(dim=-1, keepdim=True) + eps
                        target_grad = target_grad / target_norm
                        if use_neg_grad:
                            behavior_norm = behavior_grad.norm(dim=-1, keepdim=True) + eps
                            behavior_grad = behavior_grad / behavior_norm

                    if clamp:
                        target_grad = target_grad.clamp(-l_inf, l_inf)
                        if use_neg_grad:
                            behavior_grad = behavior_grad.clamp(-l_inf, l_inf)

                    if use_neg_grad:
                        guide_actions = target_grad - ratio * behavior_grad
                    else:
                        guide_actions = target_grad

                    guide = torch.zeros_like(model_mean)
                    guide[:, :, state_dim:] = guide_actions

                    if use_adaptive:
                        scale_mult = 0.5 * (1 + math.cos(math.pi * (n_timesteps - t_diff) / n_timesteps))
                        guide = scale_mult * action_scale * guide
                    else:
                        guide = action_scale * guide

                    model_mean = model_mean + guide
                    model_mean = normalize_fn(model_mean)
                    model_mean = apply_conditioning(model_mean, conditions, state_dim)
                    model_mean = unnormalize_fn(model_mean)

                model_mean = normalize_fn(model_mean)

            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_unnorm = unnormalize_fn(x)

        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store

        if total_generated >= t_gen:
            break

        last_states_norm = x[:, -1, :state_dim]
        conditions = {0: last_states_norm}

    return all_trajectories.detach().cpu().numpy()


print("Helper functions defined.")

Helper functions defined.


## Step 2: Main Loop — Evaluate Each Target Policy

For each of the 8 target policies:
1. Collect 50 rollouts (cached to disk)
2. Build training data (target rollouts + expert demos)
3. Train chunk diffuser (50 epochs)
4. Train BC_Gaussian behavior policy
5. Build target policy scorer
6. Generate synthetic trajectories with guidance (full_0.2_r0.25)
7. Score and compute OPE estimate

In [6]:
all_ope_results = {}

for policy_idx, (policy_key, run_dir) in enumerate(POLICIES.items()):
    epoch = get_epoch(policy_key)
    ckpt_path = f"models/model_epoch_{epoch}.pth"
    oracle_v = oracle_values[policy_key]["mean_return"]
    oracle_sr = oracle_values[policy_key]["success_rate"]
    
    print(f"\n{'#'*70}")
    print(f"# [{policy_idx+1}/{len(POLICIES)}] {policy_key}  (oracle V^pi={oracle_v:.4f}, SR={oracle_sr*100:.0f}%)")
    print(f"{'#'*70}")
    t0_policy = time.time()
    
    # ── Step 2a: Collect/load rollouts ──
    rollout_dir = ROLLOUT_BASE / f"multi_policy_{policy_key}"
    rollout_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(rollout_dir.glob("rollout_*.h5"))
    
    if len(existing) >= NUM_TARGET_ROLLOUTS:
        print(f"  Using {NUM_TARGET_ROLLOUTS} cached rollouts from {rollout_dir}")
    else:
        print(f"  Collecting {NUM_TARGET_ROLLOUTS} rollouts...")
        ckpt = load_checkpoint(str(run_dir), ckpt_path=ckpt_path)
        collect_rollouts(
            ckpt=ckpt,
            output_dir=rollout_dir,
            num_rollouts=NUM_TARGET_ROLLOUTS,
            horizon=T_GEN,
            obs_keys=OBS_KEYS,
            feat_type="low_dim_concat",
            device=str(device),
            verbose=True,
        )
        existing = sorted(rollout_dir.glob("rollout_*.h5"))
    
    # ── Step 2b: Load target rollouts ──
    target_data = []
    target_states_list = []
    target_actions_list = []
    
    for path in sorted(existing)[:NUM_TARGET_ROLLOUTS]:
        with h5py.File(path, "r") as f:
            latents = f["latents"][:]
            actions = f["actions"][:]
            rewards = f["rewards"][:] if "rewards" in f else np.zeros(len(actions))
        
        if latents.ndim == 3:
            states = latents[:, -1, :]
        else:
            states = latents
        
        states = states.astype(np.float32)
        actions = actions.astype(np.float32)
        
        episode = {
            "states": states[:-1],
            "actions": actions[:-1],
            "rewards": rewards[:-1] if len(rewards) > 1 else rewards,
            "next-states": states[1:],
        }
        target_data.append(episode)
        target_states_list.append(states)
        target_actions_list.append(actions)
    
    # ── Compute target SR from rollouts ──
    target_successes = [ep["states"][:, CUBE_Z_INDEX].max() > LIFT_THRESHOLD for ep in target_data]
    target_sr = np.mean(target_successes)
    print(f"  Target rollout SR: {target_sr*100:.1f}%")
    
    # ── Step 2c: Combine target + expert, compute normalization ──
    all_episodes = target_data + expert_data
    all_states = np.concatenate(target_states_list + expert_states_list, axis=0)
    all_actions = np.concatenate(target_actions_list + expert_actions_list, axis=0)
    
    norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
    norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)
    norm_mean_t = torch.tensor(norm_mean, device=device)
    norm_std_t = torch.tensor(norm_std, device=device)
    normalize_fn = lambda x, m=norm_mean_t, s=norm_std_t: (x - m) / s
    unnormalize_fn = lambda x, m=norm_mean_t, s=norm_std_t: x * s + m
    
    print(f"  Combined: {len(all_episodes)} episodes ({len(target_data)} target + {len(expert_data)} expert)")
    
    # ── Step 2d: Chunk for training ──
    chunks_states_to = []
    chunks_actions_to = []
    
    for ep in all_episodes:
        st = ep["states"]
        act = ep["actions"]
        T_ep = len(st)
        if T_ep < CHUNK_SIZE + 1:
            continue
        for t0 in range(0, T_ep - CHUNK_SIZE, STRIDE):
            s_to = st[t0 : t0 + CHUNK_SIZE + 1]
            a_to = act[t0 : t0 + CHUNK_SIZE]
            chunks_states_to.append(s_to)
            chunks_actions_to.append(a_to)
    
    chunks_states_to = np.array(chunks_states_to, dtype=np.float32)
    chunks_actions_to = np.array(chunks_actions_to, dtype=np.float32)
    
    train_x = np.concatenate([
        chunks_states_to[:, :-1, :],
        chunks_actions_to,
    ], axis=-1)
    train_cond = chunks_states_to[:, 0, :]
    
    train_x_t = torch.tensor(train_x, dtype=torch.float32, device=device)
    train_cond_t = torch.tensor(train_cond, dtype=torch.float32, device=device)
    
    print(f"  Training chunks: {len(train_x_t)}, batches/epoch: {len(train_x_t) // BATCH_SIZE}")
    
    # ── Step 2e: Train chunk diffuser ──
    temporal_model = TemporalUnet(
        horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
        dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
    ).to(device)
    
    diffusion_model = GaussianDiffusion(
        model=temporal_model, horizon=CHUNK_SIZE,
        observation_dim=STATE_DIM, action_dim=ACTION_DIM,
        n_timesteps=N_DIFFUSION_STEPS,
        normalizer=normalize_fn, unnormalizer=unnormalize_fn,
        predict_epsilon=PREDICT_EPSILON, loss_type="l2",
        clip_denoised=False, action_weight=ACTION_WEIGHT,
        loss_weights=None, loss_discount=1.0,
    ).to(device)
    
    ema = EMA(diffusion_model)
    optimizer = torch.optim.Adam(diffusion_model.parameters(), lr=LR)
    
    t0_train = time.time()
    diffusion_model.train()
    epoch_losses = []
    
    for ep_num in range(1, TRAIN_EPOCHS + 1):
        perm = torch.randperm(len(train_x_t), device=device)
        ep_loss = []
        for start in range(0, len(train_x_t) - BATCH_SIZE + 1, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            x_batch = train_x_t[idx]
            c_batch = train_cond_t[idx]
            
            x_norm = normalize_fn(x_batch)
            c_norm = normalize_fn(
                torch.cat([c_batch, torch.zeros(len(c_batch), ACTION_DIM, device=device)], dim=-1)
            )[:, :STATE_DIM]
            
            cond = {0: c_norm}
            loss, _ = diffusion_model.loss(x_norm, cond)
            
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), GRAD_CLIP)
            optimizer.step()
            ema.update(diffusion_model)
            ep_loss.append(loss.item())
        
        epoch_losses.append(np.mean(ep_loss))
    
    elapsed_train = time.time() - t0_train
    print(f"  Diffuser trained: {elapsed_train:.0f}s, final loss={epoch_losses[-1]:.6f}")
    
    # ── Step 2f: Train BC_Gaussian behavior policy ──
    demo_states = np.concatenate([ep["states"] for ep in target_data], axis=0)
    demo_actions = np.concatenate([ep["actions"] for ep in target_data], axis=0)
    
    bc_behavior = BCGaussian(STATE_DIM, ACTION_DIM, hidden_dim=BC_HIDDEN).to(device)
    bc_optimizer = torch.optim.Adam(bc_behavior.parameters(), lr=1e-3)
    states_t = torch.tensor(demo_states, dtype=torch.float32, device=device)
    actions_t = torch.tensor(demo_actions, dtype=torch.float32, device=device)
    
    bc_behavior.train()
    for bc_ep in range(BC_EPOCHS):
        idx = torch.randint(0, len(states_t), (BC_BATCH,), device=device)
        nll = -bc_behavior.log_prob(states_t[idx], actions_t[idx]).mean()
        bc_optimizer.zero_grad()
        nll.backward()
        bc_optimizer.step()
    bc_behavior.eval()
    print(f"  BC_Gaussian trained: final NLL={nll.item():.4f}")
    
    # ── Step 2g: Build target policy scorer ──
    ckpt = load_checkpoint(str(run_dir), ckpt_path=ckpt_path)
    target_algo = build_algo_from_checkpoint(ckpt, device=str(device))
    target_scorer = RobomimicDiffusionScorer(target_algo, device=str(device), obs_keys=OBS_KEYS)
    
    # ── Step 2h: Generate synthetic trajectories with guidance ──
    np.random.seed(42 + policy_idx)
    torch.manual_seed(42 + policy_idx)
    
    initial_states = get_initial_states_from_data(target_data, NUM_SYNTHETIC_TRAJS, device)
    
    trajs = generate_trajectories_full_guidance(
        diffusion_model=ema.ema_model,
        initial_states=initial_states,
        normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM, action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
        target_scorer=target_scorer,
        behavior_scorer=bc_behavior,
        action_scale=BEST_ACTION_SCALE, ratio=BEST_RATIO,
        k_guide=K_GUIDE, normalize_grad=NORMALIZE_GRAD,
        use_adaptive=USE_ADAPTIVE, clamp=CLAMP, l_inf=L_INF,
    )
    
    # ── Step 2i: Score and evaluate ──
    returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD)
    ope_estimate = float(np.mean(returns))
    ope_std = float(np.std(returns))
    rel_error = abs(ope_estimate - oracle_v) / max(abs(oracle_v), 1e-6)
    
    elapsed_policy = time.time() - t0_policy
    
    all_ope_results[policy_key] = {
        "oracle_value": oracle_v,
        "oracle_sr": oracle_sr,
        "ope_estimate": ope_estimate,
        "ope_std": ope_std,
        "ope_sr": float(np.mean(successes)),
        "rel_error": rel_error,
        "target_rollout_sr": target_sr,
        "final_loss": epoch_losses[-1],
        "elapsed_sec": elapsed_policy,
    }
    
    print(f"  >>> OPE={ope_estimate:.4f} vs Oracle={oracle_v:.4f}, rel_error={rel_error:.2%}")
    print(f"  >>> Synthetic SR={np.mean(successes)*100:.1f}%, time={elapsed_policy:.0f}s")
    
    # Clean up GPU memory
    del diffusion_model, temporal_model, ema, optimizer, target_algo, target_scorer
    del bc_behavior, bc_optimizer, train_x_t, train_cond_t, states_t, actions_t
    torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("ALL POLICIES EVALUATED")
print(f"{'='*70}")


######################################################################
# [1/4] 50demos_epoch10  (oracle V^pi=0.0000, SR=0%)
######################################################################



============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'robot0_gripper_qpos', 'robot0_eef_quat', 'object']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []


number of parameters: 6.576359e+07


[14:55:31] INFO     build_rollout_policy_from_checkpoint took 1.40 seconds to execute                 ]8;id=468192;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=516926;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\

[robosuite WARNING] No private macro file found! (macros.py:57)


...

  BC_Gaussian trained: final NLL=-8.7586



============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'robot0_gripper_qpos', 'robot0_eef_quat', 'object']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []


number of parameters: 6.576359e+07


[16:29:59] INFO     build_algo_from_checkpoint took 0.55 seconds to execute                           ]8;id=483960;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=823989;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\

  >>> OPE=0.6000 vs Oracle=0.9000, rel_error=33.33%
  >>> Synthetic SR=60.0%, time=1660s

ALL POLICIES EVALUATED


## Step 3: Results Summary

In [7]:
# ── Summary table ──
print(f"{'Policy':>20s} | {'Oracle':>7s} | {'OPE':>7s} | {'RelErr':>8s} | {'Rollout SR':>10s} | {'Synth SR':>8s}")
print("-" * 75)
for key, r in sorted(all_ope_results.items(), key=lambda x: x[1]["oracle_value"]):
    print(f"{key:>20s} | {r['oracle_value']:>7.4f} | {r['ope_estimate']:>7.4f} | "
          f"{r['rel_error']:>7.2%} | {r['target_rollout_sr']*100:>9.1f}% | {r['ope_sr']*100:>7.1f}%")

# ── Aggregate metrics ──
oracle_vals = np.array([r["oracle_value"] for r in all_ope_results.values()])
ope_vals = np.array([r["ope_estimate"] for r in all_ope_results.values()])

# Spearman rank correlation
rho, p_value = stats.spearmanr(oracle_vals, ope_vals)
print(f"\nSpearman rank correlation: rho={rho:.4f}, p={p_value:.4f}")

# MSE
mse = np.mean((oracle_vals - ope_vals) ** 2)
print(f"Mean squared error: {mse:.6f}")

# Mean absolute error
mae = np.mean(np.abs(oracle_vals - ope_vals))
print(f"Mean absolute error: {mae:.4f}")

# Regret@1: does the best OPE policy match the best oracle policy?
best_oracle_key = max(all_ope_results, key=lambda k: all_ope_results[k]["oracle_value"])
best_ope_key = max(all_ope_results, key=lambda k: all_ope_results[k]["ope_estimate"])
regret_1 = all_ope_results[best_oracle_key]["oracle_value"] - all_ope_results[best_ope_key]["oracle_value"]
print(f"\nBest oracle policy: {best_oracle_key} (V^pi={all_ope_results[best_oracle_key]['oracle_value']:.4f})")
print(f"Best OPE policy:    {best_ope_key} (V^pi={all_ope_results[best_ope_key]['oracle_value']:.4f})")
print(f"Regret@1: {regret_1:.4f}")

              Policy |  Oracle |     OPE |   RelErr | Rollout SR | Synth SR
---------------------------------------------------------------------------
     50demos_epoch10 |  0.0000 |  0.2000 | 20000000.30% |       0.0% |    20.0%
    200demos_epoch10 |  0.1800 |  0.0000 | 100.00% |       0.0% |     0.0%
     10demos_epoch20 |  0.5200 |  0.0500 |  90.38% |       0.0% |     5.0%
    200demos_epoch40 |  0.9000 |  0.6000 |  33.33% |       0.0% |    60.0%

Spearman rank correlation: rho=0.4000, p=0.6000
Mean squared error: 0.095825
Mean absolute error: 0.2875

Best oracle policy: 200demos_epoch40 (V^pi=0.9000)
Best OPE policy:    200demos_epoch40 (V^pi=0.9000)
Regret@1: 0.0000


## Step 4: Visualizations

In [8]:
# ── Figure 1: Oracle vs OPE scatter plot ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter
ax = axes[0]
for key, r in all_ope_results.items():
    ax.scatter(r["oracle_value"], r["ope_estimate"], s=80, zorder=5)
    ax.annotate(key, (r["oracle_value"], r["ope_estimate"]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)

lims = [0, 1]
ax.plot(lims, lims, "k--", alpha=0.3, label="ideal (y=x)")
ax.set_xlabel("Oracle V^pi")
ax.set_ylabel("OPE Estimate")
ax.set_title(f"Oracle vs OPE (Spearman rho={rho:.3f}, p={p_value:.3f})")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")

# Bar chart: oracle vs OPE side by side
ax = axes[1]
sorted_keys = sorted(all_ope_results.keys(), key=lambda k: all_ope_results[k]["oracle_value"])
x_pos = np.arange(len(sorted_keys))
width = 0.35
oracle_bars = [all_ope_results[k]["oracle_value"] for k in sorted_keys]
ope_bars = [all_ope_results[k]["ope_estimate"] for k in sorted_keys]
ax.bar(x_pos - width/2, oracle_bars, width, label="Oracle", color="steelblue")
ax.bar(x_pos + width/2, ope_bars, width, label="OPE", color="coral")
ax.set_xticks(x_pos)
ax.set_xticklabels([k.replace("demos_epoch", "d_e") for k in sorted_keys], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Value")
ax.set_title("Oracle vs OPE by Policy")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Relative error bar chart
ax = axes[2]
rel_errors = [all_ope_results[k]["rel_error"] for k in sorted_keys]
colors = ["green" if e < 0.25 else "orange" if e < 0.5 else "red" for e in rel_errors]
ax.bar(x_pos, rel_errors, color=colors)
ax.set_xticks(x_pos)
ax.set_xticklabels([k.replace("demos_epoch", "d_e") for k in sorted_keys], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Relative Error")
ax.set_title("Relative Error by Policy")
ax.axhline(y=0.25, color="green", linestyle="--", alpha=0.5, label="25%")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="50%")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("MVP v0.3.2: Multi-Policy OPE Evaluation", fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

In [9]:
print(f"\n{'='*70}")
print("MVP v0.3.2 COMPLETE — Multi-Policy Positive and Negative Test")
print(f"{'='*70}")
print(f"Policies evaluated: {len(all_ope_results)}")
print(f"Spearman rho: {rho:.4f} (p={p_value:.4f})")
print(f"Regret@1: {regret_1:.4f}")
print(f"Mean rel error: {np.mean([r['rel_error'] for r in all_ope_results.values() if r['oracle_value'] > 0]):.2%} (excl. 0% oracle policies)")
print(f"MSE: {mse:.6f}")
print(f"MAE: {mae:.4f}")
print()
print("Positive test (rho > 0.7): " + ("PASS" if rho > 0.7 else "FAIL"))
print("Negative test (0% policy OPE < 0.1): ", end="")
zero_key = "50demos_epoch10"
if zero_key in all_ope_results:
    zero_ope = all_ope_results[zero_key]["ope_estimate"]
    print(f"{'PASS' if zero_ope < 0.1 else 'FAIL'} (OPE={zero_ope:.4f})")
print()
total_time = sum(r["elapsed_sec"] for r in all_ope_results.values())
print(f"Total wall time: {total_time/3600:.1f} hours")


MVP v0.3.2 COMPLETE — Multi-Policy Positive and Negative Test
Policies evaluated: 4
Spearman rho: 0.4000 (p=0.6000)
Regret@1: 0.0000
Mean rel error: 74.57% (excl. 0% oracle policies)
MSE: 0.095825
MAE: 0.2875

Positive test (rho > 0.7): FAIL
Negative test (0% policy OPE < 0.1): FAIL (OPE=0.2000)

Total wall time: 1.6 hours
